In [1]:
import pandas as pd

# Define the path to your raw file
raw_file_path = 'data/Forecasting Case- Study.csv'

# Load the raw data
df_raw = pd.read_csv(raw_file_path)

# Create a WORKING COPY for our cleaning process
# This ensures df_raw stays pure
df = df_raw.copy()

In [2]:
print(f"Dataset Shape: {df.shape}")
print("\n--- First 5 Rows ---")
print(df.head())

Dataset Shape: (8084, 4)

--- First 5 Rows ---
        State       Date           Total   Category
0     Alabama  1/12/2019    109,574,036   Beverages
1     Arizona  1/12/2019    109,101,595   Beverages
2    Arkansas  1/12/2019     58,049,432   Beverages
3  California  1/12/2019    444,766,891   Beverages
4    Colorado  1/12/2019     89,816,716   Beverages


In [3]:
print("\n--- Column Info & Data Types ---")
df.info()


--- Column Info & Data Types ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8084 entries, 0 to 8083
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   State     8084 non-null   object
 1   Date      8084 non-null   object
 2   Total     8084 non-null   object
 3   Category  8084 non-null   object
dtypes: object(4)
memory usage: 252.8+ KB


In [4]:
print("\n--- Missing Values Per Column ---")
print(df.isnull().sum())


--- Missing Values Per Column ---
State       0
Date        0
Total       0
Category    0
dtype: int64


In [6]:
# 1. Clean the 'Total' column: Remove commas and turn into decimal numbers
df['Total'] = df['Total'].replace({',': ''}, regex=True).astype(float)

In [7]:
# 2. Convert 'Date': dayfirst=True is safer for international formats
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')

In [8]:
# 3. Handle the "Dropping" safety check
# If any dates failed to convert, they become 'NaT'. We remove them now.
df = df.dropna(subset=['Date'])

In [9]:
# 4. Sort chronologically so history flows from past to present
df = df.sort_values(by=['State', 'Date'])

In [10]:
print("Data Types after cleaning:")
print(df.dtypes)

Data Types after cleaning:
State               object
Date        datetime64[ns]
Total              float64
Category            object
dtype: object


Gap Filling (Resampling)

In [14]:
def fill_missing_weeks(state_df):
    
    full_range = pd.date_range(start=state_df['Date'].min(), 
                               end=state_df['Date'].max(), 
                               freq='W-SUN')
    
    
    state_df = state_df.set_index('Date').reindex(full_range).reset_index()
    state_df.rename(columns={'index': 'Date'}, inplace=True)
    
    
    state_df['Total'] = state_df['Total'].fillna(0)
    state_df['Category'] = state_df['Category'].ffill().bfill()
    
    return state_df


df_complete = df.groupby('State', group_keys=True).apply(fill_missing_weeks, include_groups=False).reset_index()


df_complete.rename(columns={'level_0': 'State'}, inplace=True)

print(f"Original Row Count: {len(df)}")
print(f"New Row Count (Gaps filled): {len(df_complete)}")
print(df_complete.head())

Original Row Count: 3225
New Row Count (Gaps filled): 8084
     State  level_1       Date        Total   Category
0  Alabama        0 2019-10-06  129106730.0  Beverages
1  Alabama        1 2019-10-13          0.0  Beverages
2  Alabama        2 2019-10-20          0.0  Beverages
3  Alabama        3 2019-10-27          0.0  Beverages
4  Alabama        4 2019-11-03  112189104.0  Beverages


In [15]:
# 1. Create Lag features (t-1, t-7, t-30)
# These tell the model what happened in the past
df_complete['lag_1'] = df_complete.groupby('State')['Total'].shift(1)
df_complete['lag_7'] = df_complete.groupby('State')['Total'].shift(7)
df_complete['lag_30'] = df_complete.groupby('State')['Total'].shift(30)

# 2. Rolling Mean (4 weeks)
# This shows the average sales of the last month
df_complete['rolling_mean_4'] = df_complete.groupby('State')['Total'].shift(1).rolling(window=4).mean()

# 3. Date Components for Seasonality
df_complete['month'] = df_complete['Date'].dt.month
df_complete['week_of_year'] = df_complete['Date'].dt.isocalendar().week

# 4. Remove the rows at the very beginning that don't have enough history for lags
df_final = df_complete.dropna()

# 5. Save the FINAL processed file
df_final.to_csv('data/final_processed_data.csv', index=False)

print(f"Final Step Complete! Dataset is ready for modeling.")
print(f"Final shape: {df_final.shape}")

Final Step Complete! Dataset is ready for modeling.
Final shape: (6794, 11)
